# Tumulus LiDAR Detector

Detects burial mounds (tumuli) in Romania's 0.5 m airborne LiDAR, in your browser. No install.

## To start: open the Runtime menu, click Run all, and wait about a minute.

It sets up, shows the map below, and scans a default area automatically. The blue area is where the model can scan (0.5 m LiDAR, ANCPI LAKI III; mostly Oltenia and SW Romania).

<img src="https://raw.githubusercontent.com/ObuObuHub/tumulus-lidar-detector/main/assets/coverage_preview.png" width="460">

To scan somewhere else: click the map to read a point's coordinates, type them into the Scan cell, and run it. Form is not proof: only fieldwork confirms a tumulus. Repo: https://github.com/ObuObuHub/tumulus-lidar-detector

In [ ]:
#@title Setup (run once) { display-mode: "form" }
import os
if not os.path.isdir('tumulus-lidar-detector') and os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    !git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
if os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    %cd tumulus-lidar-detector
!pip install -q pyproj folium
print("Ready. See the map below, then run the Scan cell.")

In [ ]:
#@title Map of the scannable area - click anywhere to read its coordinates { display-mode: "form" }
import json, folium
_cov = json.load(open('assets/laki3_coverage.geojson'))
_m = folium.Map(location=[44.6, 23.4], zoom_start=7, tiles='OpenStreetMap', control_scale=True)
folium.GeoJson(_cov, name='0.5 m LiDAR coverage',
               style_function=lambda _f: {'color': '#1d4ed8', 'fillColor': '#2563eb',
                                          'fillOpacity': 0.30, 'weight': 1}).add_to(_m)
folium.Marker([44.043, 23.522], tooltip='Default scan point (Catane)',
              icon=folium.Icon(color='red', icon='info-sign')).add_to(_m)
_m.add_child(folium.LatLngPopup())   # click the map -> popup shows that point's lat/lon
_m

In [ ]:
#@title Scan { display-mode: "form" }
#@markdown Default scans Catane. To scan elsewhere: click the map above to read a point's lat/lon, type them here, and run.
longitude = 23.522  #@param {type:"number"}
latitude  = 44.043  #@param {type:"number"}
area_km   = 1       #@param {type:"slider", min:1, max:5, step:1}

import os, json
# clear any previous run's outputs so a new scan can never show stale results
for _f in ('/tmp/zone_dets.csv', 'review/zone_view.jpg', 'review/zone_board.jpg', 'detected_mounds.csv'):
    if os.path.exists(_f):
        os.remove(_f)

lon, lat = longitude, latitude

def _inside(lon, lat):
    cov = json.load(open('assets/laki3_coverage.geojson'))
    for f in cov['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

print("Scan point: lat %s, lon %s  |  box %s km" % (lat, lon, area_km))
if not _inside(lon, lat):
    print("\nThis point is OUTSIDE the 0.5 m LiDAR coverage (the blue area on the map).")
    print("Pick a point inside the blue and run this cell again.")
else:
    !python tools/scan_zone.py {lon} {lat} {area_km}

In [ ]:
#@title Results { display-mode: "form" }
import os, pandas as pd
from IPython.display import Image, HTML, display

if not os.path.exists('review/zone_view.jpg'):
    print("No result yet. Run the Scan cell above on a point inside the blue coverage.")
else:
    n = 0
    if os.path.exists('/tmp/zone_dets.csv'):
        det = pd.read_csv('/tmp/zone_dets.csv')
        kept = det[det['keep'] == 1] if 'keep' in det.columns else det.iloc[0:0]
        n = len(kept)
    if n > 0:
        print("Found %d probable mound(s). Coordinates and CSV below; green circles on the map." % n)
    else:
        print("No mounds found in this area. This is normal for most points: the model is selective.")
    display(Image('review/zone_view.jpg'))
    if os.path.exists('review/zone_board.jpg'):
        display(Image('review/zone_board.jpg'))
    if n > 0:
        m = kept[['lon', 'lat', 'score', 'coh', 'pgate']].sort_values('score', ascending=False).reset_index(drop=True)
        m.insert(0, 'id', range(1, len(m) + 1))
        show = m.copy()
        show['map'] = show.apply(lambda r: '<a href="https://www.google.com/maps?q=%s,%s" target="_blank">view</a>' % (r.lat, r.lon), axis=1)
        display(HTML(show.to_html(escape=False, index=False)))
        m.to_csv('detected_mounds.csv', index=False)
        try:
            from google.colab import files
            files.download('detected_mounds.csv')
        except Exception:
            print("detected_mounds.csv saved; see the Files panel on the left.")